# 02 - Análisis Exploratorio de Datos - Catastro DNC

En este notebook se realiza el análisis exploratorio de datos sobre las tablas refinadas de la Dirección Nacional de Catastro.

Los datos ya fueron ingestados mediante NiFi, almacenados en HDFS y procesados en el notebook anterior, donde se asignaron los cabezales definidos en los metadatos oficiales y se guardaron las tablas en formato Parquet en la zona `/ref`.

El objetivo de este notebook es analizar la estructura y calidad de los datos refinados, revisando:

- Cantidad de registros y columnas por tabla.
- Esquemas de datos.
- Vista previa de registros.
- Valores nulos o faltantes.
- Registros duplicados.
- Posibles claves primarias.
- Reglas de negocio relevantes para el refinamiento y modelado posterior.

Todo el análisis se realiza utilizando PySpark.

## 1. Inicialización del entorno Spark

Se inicializa la sesión de Spark con soporte para Hive, ya que más adelante se utilizarán tablas externas para consultar el modelo analítico.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col,
    count,
    countDistinct,
    when,
    isnan,
    trim,
    lit,
    desc,
    asc,
    min as spark_min,
    max as spark_max,
    avg,
    sum as spark_sum
)

spark = SparkSession.builder \
    .appName("Obligatorio Catastro - EDA") \
    .enableHiveSupport() \
    .getOrCreate()

spark

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


2026-06-05T02:24:27,377 WARN [Thread-4] org.apache.hadoop.util.NativeCodeLoader - Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
2026-06-05T02:24:28,545 WARN [Thread-4] org.apache.spark.util.Utils - Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


## 2. Definición de rutas y tablas refinadas

Las tablas utilizadas en este análisis se encuentran en la zona refinada del datalake.  
Esta zona contiene los datos en formato Parquet, con nombres de columnas asignados según los metadatos oficiales.

In [2]:
REF_PATH = "/ref/obligatorio_catastro/dnc_2026_05"

tablas_ref = [
    "departamentos",
    "localidades",
    "destinos",
    "categorias_de_construccion",
    "estados_de_conservacion",
    "cubiertas",
    "cielorrasos",
    "tipos_de_obra",
    "padrones_urbanos",
    "padrones_rurales",
    "lineas_de_construccion",
    "mutaciones_catastrales",
    "historico_de_valores"
]

print("Ruta REF:", REF_PATH)
print("Cantidad de tablas refinadas:", len(tablas_ref))

for tabla in tablas_ref:
    print("-", tabla)

Ruta REF: /ref/obligatorio_catastro/dnc_2026_05
Cantidad de tablas refinadas: 13
- departamentos
- localidades
- destinos
- categorias_de_construccion
- estados_de_conservacion
- cubiertas
- cielorrasos
- tipos_de_obra
- padrones_urbanos
- padrones_rurales
- lineas_de_construccion
- mutaciones_catastrales
- historico_de_valores


## 3. Carga de tablas refinadas

Se cargan todas las tablas Parquet desde HDFS en un diccionario de DataFrames de Spark.  
Esto permite recorrerlas de forma dinámica durante el análisis exploratorio.

In [3]:
dfs = {}

for tabla in tablas_ref:
    path = f"{REF_PATH}/{tabla}"
    dfs[tabla] = spark.read.parquet(path)
    print(f"Tabla cargada: {tabla}")

Tabla cargada: departamentos
Tabla cargada: localidades
Tabla cargada: destinos
Tabla cargada: categorias_de_construccion
Tabla cargada: estados_de_conservacion
Tabla cargada: cubiertas
Tabla cargada: cielorrasos
Tabla cargada: tipos_de_obra
Tabla cargada: padrones_urbanos
Tabla cargada: padrones_rurales
Tabla cargada: lineas_de_construccion
Tabla cargada: mutaciones_catastrales
Tabla cargada: historico_de_valores


## 4. Resumen general de tablas refinadas

En esta sección se genera un resumen general de las tablas refinadas, indicando para cada una:

- Cantidad de registros.
- Cantidad de columnas.
- Nombres de columnas.

Este resumen permite validar que las 13 tablas esperadas fueron cargadas correctamente desde la zona refinada.

In [4]:
from pyspark.sql import Row

resumen_tablas = []

for tabla, df in dfs.items():
    resumen_tablas.append(
        Row(
            tabla=tabla,
            cantidad_registros=df.count(),
            cantidad_columnas=len(df.columns),
            columnas=", ".join(df.columns)
        )
    )

df_resumen_tablas = spark.createDataFrame(resumen_tablas)

df_resumen_tablas.orderBy("tabla").show(truncate=False)

+--------------------------+------------------+-----------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|tabla                     |cantidad_registros|cantidad_columnas|columnas                                                                                                                                                                                                                                                                                                          |
+--------------------------+------------------+-----------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

### Observación inicial

Las tablas refinadas fueron cargadas correctamente desde HDFS.  
El conjunto contiene tablas maestras pequeñas, como departamentos, localidades y tablas auxiliares de construcción, y tablas transaccionales o de mayor volumen, como padrones urbanos, líneas de construcción e histórico de valores.

Esta diferencia de volumen es relevante para el análisis posterior, ya que las tablas grandes concentran los hechos principales del dominio catastral, mientras que las tablas pequeñas funcionan como dimensiones o catálogos descriptivos.

## 5. Vista previa y esquema de cada tabla

En esta sección se muestran las primeras filas y el esquema de cada tabla refinada.  
Esto permite validar que los encabezados asignados en la etapa anterior coinciden con la estructura esperada según los metadatos de la Dirección Nacional de Catastro.

In [5]:
for tabla in tablas_ref:
    print("=" * 100)
    print(f"TABLA: {tabla}")
    print("=" * 100)
    
    df = dfs[tabla]
    
    print("Columnas:")
    print(df.columns)
    
    print("\nEsquema:")
    df.printSchema()
    
    print("\nPrimeras 5 filas:")
    df.show(5, truncate=False)

TABLA: departamentos
Columnas:
['codigo_departamento', 'departamento']

Esquema:
root
 |-- codigo_departamento: string (nullable = true)
 |-- departamento: string (nullable = true)


Primeras 5 filas:
+-------------------+--------------+
|codigo_departamento|departamento  |
+-------------------+--------------+
|A                  |CANELONES     |
|B                  |MALDONADO     |
|C                  |ROCHA         |
|D                  |TREINTA Y TRES|
|E                  |CERRO LARGO   |
+-------------------+--------------+
only showing top 5 rows

TABLA: localidades
Columnas:
['codigo_departamento', 'codigo_localidad', 'localidad']

Esquema:
root
 |-- codigo_departamento: string (nullable = true)
 |-- codigo_localidad: string (nullable = true)
 |-- localidad: string (nullable = true)


Primeras 5 filas:
+-------------------+----------------+-----------+
|codigo_departamento|codigo_localidad|localidad  |
+-------------------+----------------+-----------+
|A                  |AA    

### Observación sobre los esquemas

En esta etapa las tablas refinadas ya cuentan con nombres de columnas asignados.  
Algunas columnas numéricas aún pueden aparecer como tipo `string`, ya que en el primer procesamiento se priorizó conservar los datos originales con sus encabezados correctos.  

Durante el análisis de calidad y refinamiento posterior se identificarán las columnas que deben convertirse a tipos numéricos o fecha para el modelo analítico.

## 6. Análisis de valores nulos y faltantes

En esta sección se revisa la presencia de valores nulos o vacíos en cada columna de las tablas refinadas.

Este análisis permite identificar problemas de calidad de datos, columnas opcionales, campos incompletos y posibles reglas de limpieza para etapas posteriores del datalake.

In [6]:
def analizar_nulos(df, tabla):
    """
    Calcula cantidad y porcentaje de valores nulos o vacíos por columna.
    Se consideran faltantes:
    - NULL
    - string vacío
    """
    total_registros = df.count()
    
    expresiones = []
    
    for c in df.columns:
        expresiones.append(
            count(
                when(
                    col(c).isNull() | (trim(col(c)) == ""),
                    c
                )
            ).alias(c)
        )
    
    df_nulos = df.select(expresiones)
    
    filas = []
    resultado = df_nulos.collect()[0].asDict()
    
    for columna, cantidad_nulos in resultado.items():
        porcentaje = 0 if total_registros == 0 else (cantidad_nulos / total_registros) * 100
        
        filas.append(
            Row(
                tabla=tabla,
                columna=columna,
                cantidad_registros=total_registros,
                cantidad_nulos_o_vacios=cantidad_nulos,
                porcentaje_nulos_o_vacios=round(porcentaje, 2)
            )
        )
    
    return spark.createDataFrame(filas)
    

In [7]:
df_nulos_general = None

for tabla in tablas_ref:
    print(f"Analizando nulos en: {tabla}")
    
    df_nulos_tabla = analizar_nulos(dfs[tabla], tabla)
    
    if df_nulos_general is None:
        df_nulos_general = df_nulos_tabla
    else:
        df_nulos_general = df_nulos_general.unionByName(df_nulos_tabla)

df_nulos_general.orderBy(
    desc("porcentaje_nulos_o_vacios"),
    asc("tabla"),
    asc("columna")
).show(200, truncate=False)

Analizando nulos en: departamentos
Analizando nulos en: localidades
Analizando nulos en: destinos
Analizando nulos en: categorias_de_construccion
Analizando nulos en: estados_de_conservacion
Analizando nulos en: cubiertas
Analizando nulos en: cielorrasos
Analizando nulos en: tipos_de_obra
Analizando nulos en: padrones_urbanos


Analizando nulos en: padrones_rurales
Analizando nulos en: lineas_de_construccion


Analizando nulos en: mutaciones_catastrales
Analizando nulos en: historico_de_valores


[Stage 144:====================================================>  (25 + 1) / 26]

+--------------------------+-----------------------+------------------+-----------------------+-------------------------+
|tabla                     |columna                |cantidad_registros|cantidad_nulos_o_vacios|porcentaje_nulos_o_vacios|
+--------------------------+-----------------------+------------------+-----------------------+-------------------------+
|historico_de_valores      |ep_ss                  |1668657           |1668657                |100.0                    |
|mutaciones_catastrales    |fecha_vigencia         |1943              |1943                   |100.0                    |
|mutaciones_catastrales    |referencia             |1943              |1943                   |100.0                    |
|lineas_de_construccion    |ep_ss_uso_exclusivo    |4478065           |4476731                |99.97                    |
|lineas_de_construccion    |ep_ss                  |4478065           |4466102                |99.73                    |
|padrones_urbanos       

### Observaciones sobre valores faltantes

El análisis de nulos permite distinguir entre campos obligatorios y campos opcionales.  
En particular, en datos catastrales existen campos que pueden no aplicar para todos los registros, como información de entrepiso/subsuelo, block/manzana, unidades de propiedad horizontal, fechas de DJCU o datos asociados a uso exclusivo.

Estos valores no necesariamente representan errores, sino que deben interpretarse según el régimen del padrón y las características del inmueble.

## 7. Análisis de registros duplicados

En esta sección se revisa si existen filas completamente duplicadas en cada tabla refinada.

Un registro duplicado completo es aquel donde todos los valores de todas las columnas coinciden con otro registro de la misma tabla. Detectar duplicados es importante para evaluar la calidad de los datos antes de construir el modelo analítico.

In [8]:
duplicados_tablas = []

for tabla, df in dfs.items():
    total = df.count()
    distintos = df.dropDuplicates().count()
    duplicados = total - distintos
    
    duplicados_tablas.append(
        Row(
            tabla=tabla,
            cantidad_registros=total,
            cantidad_registros_distintos=distintos,
            cantidad_duplicados=duplicados,
            porcentaje_duplicados=round((duplicados / total) * 100, 4) if total > 0 else 0
        )
    )

df_duplicados_tablas = spark.createDataFrame(duplicados_tablas)

df_duplicados_tablas.orderBy(desc("cantidad_duplicados")).show(truncate=False)

[Stage 258:===================>                                     (1 + 2) / 3]

+--------------------------+------------------+----------------------------+-------------------+---------------------+
|tabla                     |cantidad_registros|cantidad_registros_distintos|cantidad_duplicados|porcentaje_duplicados|
+--------------------------+------------------+----------------------------+-------------------+---------------------+
|lineas_de_construccion    |4478065           |4010953                     |467112             |10.4311              |
|mutaciones_catastrales    |1943              |1873                        |70                 |3.6027               |
|departamentos             |19                |19                          |0                  |0.0                  |
|cielorrasos               |1                 |1                           |0                  |0.0                  |
|localidades               |498               |498                         |0                  |0.0                  |
|tipos_de_obra             |26                |2

### Observación sobre duplicados

El análisis anterior permite detectar duplicados exactos a nivel de fila completa.  
En caso de existir duplicados, se deberá decidir si corresponden a errores de carga, registros repetidos en origen o situaciones válidas del dominio.

Además del análisis de duplicados completos, es necesario validar claves primarias candidatas, ya que puede haber duplicados según la clave lógica aunque las filas no sean exactamente iguales.

## 8. Validación de claves primarias candidatas

A partir de los metadatos de la Dirección Nacional de Catastro, se definen claves candidatas para las tablas principales y auxiliares.

La validación consiste en comparar la cantidad total de registros contra la cantidad de combinaciones distintas de los campos que componen la clave candidata. Si ambos valores coinciden, la clave candidata identifica de forma única cada registro.

In [9]:
claves_candidatas = {
    "departamentos": ["codigo_departamento"],
    "localidades": ["codigo_departamento", "codigo_localidad"],
    "destinos": ["codigo_destino"],
    "categorias_de_construccion": ["codigo_categoria"],
    "estados_de_conservacion": ["codigo_estado"],
    "cubiertas": ["codigo_cubierta"],
    "cielorrasos": ["codigo_cielorraso"],
    "tipos_de_obra": ["codigo_tipo_obra"],
    
    # En padrones urbanos la identificación lógica depende de régimen, ubicación y unidad.
    "padrones_urbanos": [
        "codigo_regimen",
        "codigo_departamento",
        "codigo_localidad",
        "nro_padron",
        "block_manzana",
        "ep_ss",
        "unidad"
    ],
    
    # En padrones rurales la identificación depende del departamento, sección y padrón.
    "padrones_rurales": [
        "codigo_departamento",
        "seccion_catastral",
        "nro_padron"
    ],
    
    # Las líneas de construcción pueden tener más de una línea para un mismo padrón.
    # Se agrega la información constructiva para validar duplicados lógicos.
    "lineas_de_construccion": [
        "codigo_regimen",
        "codigo_departamento",
        "codigo_localidad",
        "nro_padron",
        "block_manzana",
        "ep_ss",
        "unidad",
        "nivel",
        "codigo_destino",
        "codigo_categoria",
        "codigo_estado",
        "codigo_cubierta",
        "codigo_cielorraso",
        "codigo_tipo_obra",
        "area_construida",
        "anio_construccion",
        "anio_remanente"
    ],
    
    # En mutaciones puede haber varios eventos sobre un mismo padrón.
    "mutaciones_catastrales": [
        "codigo_regimen",
        "codigo_departamento",
        "seccion_catastral",
        "codigo_localidad",
        "nro_padron",
        "block_manzana",
        "ep_ss",
        "unidad",
        "fecha_vigencia",
        "nro_padron_origen",
        "referencia"
    ],
    
    # Histórico de valores se valida a nivel de identificador del padrón.
    "historico_de_valores": [
        "codigo_regimen",
        "codigo_departamento",
        "seccion_catastral",
        "codigo_localidad",
        "nro_padron",
        "block_manzana",
        "ep_ss",
        "unidad"
    ]
}

In [10]:
validacion_claves = []

for tabla, columnas_clave in claves_candidatas.items():
    df = dfs[tabla]
    
    total = df.count()
    distintos_clave = df.select(*columnas_clave).dropDuplicates().count()
    duplicados_clave = total - distintos_clave
    
    validacion_claves.append(
        Row(
            tabla=tabla,
            columnas_clave=", ".join(columnas_clave),
            cantidad_registros=total,
            combinaciones_distintas_clave=distintos_clave,
            duplicados_por_clave=duplicados_clave,
            clave_unica=(duplicados_clave == 0)
        )
    )

df_validacion_claves = spark.createDataFrame(validacion_claves)

df_validacion_claves.orderBy("tabla").show(truncate=False)

[Stage 376:>                                                        (0 + 2) / 2]

+--------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------------+-----------------------------+--------------------+-----------+
|tabla                     |columnas_clave                                                                                                                                                                                                                                                   |cantidad_registros|combinaciones_distintas_clave|duplicados_por_clave|clave_unica|
+--------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

### Observación sobre claves candidatas

Las claves candidatas permiten validar la granularidad de cada tabla.  
En las tablas auxiliares se espera que el código identificador sea único. En las tablas principales, como padrones urbanos, rurales, líneas de construcción e histórico de valores, las claves son compuestas.

Si alguna clave candidata presenta duplicados, esto no implica necesariamente un error; puede indicar que la granularidad real de la tabla requiere más campos, o que existen eventos múltiples para una misma entidad catastral.

### Interpretación de la validación de claves

La validación muestra que las tablas auxiliares y las tablas principales de padrones tienen claves candidatas únicas.

En particular, `padrones_urbanos`, `padrones_rurales` e `historico_de_valores` no presentan duplicados según las claves candidatas definidas, por lo que esas combinaciones identifican correctamente cada registro.

En cambio, `lineas_de_construccion` y `mutaciones_catastrales` presentan duplicados según las claves candidatas propuestas. Esto indica que la granularidad de esas tablas no queda completamente identificada por los campos seleccionados.

Para `lineas_de_construccion`, pueden existir varias líneas con las mismas características constructivas asociadas a un mismo padrón o unidad. Por lo tanto, para el modelo analítico se tratará como una tabla de hechos de construcciones, donde cada fila representa una línea constructiva registrada, sin asumir que la clave candidata propuesta sea una clave primaria estricta.

Para `mutaciones_catastrales`, pueden existir múltiples eventos o registros asociados a un mismo padrón, fecha o referencia. Por este motivo, se interpretará como una tabla de eventos catastrales, cuya granularidad deberá analizarse con mayor detalle si se utiliza en el modelo final.

In [11]:
# Ejemplos de claves duplicadas en líneas de construcción

clave_lineas = claves_candidatas["lineas_de_construccion"]

df_lineas_duplicadas = (
    dfs["lineas_de_construccion"]
    .groupBy(*clave_lineas)
    .count()
    .filter(col("count") > 1)
    .orderBy(desc("count"))
)

df_lineas_duplicadas.show(10, truncate=False)

[Stage 381:>                                                        (0 + 2) / 2]

2026-06-05T02:27:27,093 WARN [Executor task launch worker for task 1.0 in stage 381.0 (TID 306)] org.apache.spark.sql.catalyst.expressions.RowBasedKeyValueBatch - Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
2026-06-05T02:27:27,170 WARN [Executor task launch worker for task 0.0 in stage 381.0 (TID 305)] org.apache.spark.sql.catalyst.expressions.RowBasedKeyValueBatch - Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
2026-06-05T02:27:30,338 WARN [Executor task launch worker for task 1.0 in stage 381.0 (TID 306)] org.apache.spark.sql.catalyst.expressions.RowBasedKeyValueBatch - Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
2026-06-05T02:27:30,784 WARN [Executor task launch worker for task 0.0 in stage 381.0 (TID 305)] org.apache.spark.sql.catalyst.expressions.RowBasedKeyValueBatch - Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
2026-06-05T02:27:33,687 WARN [Executor task launch worker for ta

[Stage 383:======================================>                  (2 + 1) / 3]

+--------------+-------------------+----------------+----------+-------------+-----+------+-----+--------------+----------------+-------------+---------------+-----------------+----------------+---------------+-----------------+--------------+-----+
|codigo_regimen|codigo_departamento|codigo_localidad|nro_padron|block_manzana|ep_ss|unidad|nivel|codigo_destino|codigo_categoria|codigo_estado|codigo_cubierta|codigo_cielorraso|codigo_tipo_obra|area_construida|anio_construccion|anio_remanente|count|
+--------------+-------------------+----------------+----------+-------------+-----+------+-----+--------------+----------------+-------------+---------------+-----------------+----------------+---------------+-----------------+--------------+-----+
|CO            |V                  |AA              |100947    |null         |null |0     |0.0  |1             |4.0             |3.0          |2              |0                |0               |29             |1996             |0             |261  |


In [12]:
# Ejemplos de claves duplicadas en mutaciones catastrales

clave_mutaciones = claves_candidatas["mutaciones_catastrales"]

df_mutaciones_duplicadas = (
    dfs["mutaciones_catastrales"]
    .groupBy(*clave_mutaciones)
    .count()
    .filter(col("count") > 1)
    .orderBy(desc("count"))
)

df_mutaciones_duplicadas.show(10, truncate=False)

+--------------+-------------------+-----------------+----------------+----------+-------------+-----+------+--------------+-----------------+----------+-----+
|codigo_regimen|codigo_departamento|seccion_catastral|codigo_localidad|nro_padron|block_manzana|ep_ss|unidad|fecha_vigencia|nro_padron_origen|referencia|count|
+--------------+-------------------+-----------------+----------------+----------+-------------+-----+------+--------------+-----------------+----------+-----+
|PH            |A                  |0                |WA              |47567     |A            |null |1     |null          |21/04/2026       |null      |5    |
|PH            |A                  |0                |WA              |41795     |null         |null |1     |null          |27/04/2026       |null      |5    |
|PH            |A                  |0                |WA              |48790     |A            |null |101   |null          |09/04/2026       |null      |4    |
|PH            |I                  |0   

### Análisis de duplicados por clave candidata

En la tabla `lineas_de_construccion` se observan combinaciones repetidas según la clave candidata propuesta. Esto indica que la tabla no tiene una clave primaria simple identificable con los campos disponibles en los metadatos. Para el análisis posterior se interpretará como una tabla de hechos donde cada fila representa una línea constructiva registrada, pudiendo existir múltiples registros con las mismas características para un mismo padrón.

En la tabla `mutaciones_catastrales` también se observan duplicados según la clave candidata propuesta. Además, se detecta que algunos campos no fueron interpretados correctamente como fecha en la etapa anterior, por ejemplo `fecha_vigencia` aparece como nula y `nro_padron_origen` contiene valores con formato de fecha. Esto deberá revisarse antes de utilizar esta tabla en el modelo analítico final.

Dado que el foco principal del análisis estará en padrones, construcciones, valores catastrales y dimensiones territoriales, las mutaciones catastrales se conservarán como tabla refinada disponible, pero no serán utilizadas como tabla principal del modelo analítico.

## 9. Estadísticas básicas de variables numéricas

En esta sección se analizan variables numéricas relevantes para el dominio catastral, como áreas, valores catastrales, años de construcción y métricas asociadas a los padrones.

Como las tablas fueron cargadas inicialmente desde CSV sin inferencia de tipos, algunas columnas numéricas pueden estar almacenadas como `string`. Para este análisis se realizan conversiones temporales dentro del notebook de EDA, sin modificar aún los datos refinados.

In [13]:
from pyspark.sql.functions import regexp_replace
from pyspark.sql.types import LongType, IntegerType, DoubleType

def convertir_columnas_numericas(df, columnas_long=None, columnas_int=None, columnas_double=None):
    columnas_long = columnas_long or []
    columnas_int = columnas_int or []
    columnas_double = columnas_double or []
    
    for c in columnas_long:
        df = df.withColumn(c, col(c).cast(LongType()))
        
    for c in columnas_int:
        df = df.withColumn(c, col(c).cast(IntegerType()))
        
    for c in columnas_double:
        df = df.withColumn(c, col(c).cast(DoubleType()))
        
    return df

In [14]:
df_padrones_urbanos_eda = convertir_columnas_numericas(
    dfs["padrones_urbanos"],
    columnas_long=[
        "area_predio",
        "area_edificada",
        "valor_catastral_terreno",
        "valor_catastral_mejoras",
        "valor_catastral_total",
        "valor_para_impuestos"
    ],
    columnas_int=[
        "nro_padron",
        "unidad"
    ]
)

df_padrones_urbanos_eda.select(
    "area_predio",
    "area_edificada",
    "valor_catastral_terreno",
    "valor_catastral_mejoras",
    "valor_catastral_total",
    "valor_para_impuestos"
).summary("count", "min", "max", "mean", "stddev").show(truncate=False)

2026-06-05T02:28:05,142 WARN [Thread-4] org.apache.spark.sql.catalyst.util.package - Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


[Stage 387:>                                                        (0 + 2) / 2]

+-------+------------------+------------------+-----------------------+-----------------------+---------------------+--------------------+
|summary|area_predio       |area_edificada    |valor_catastral_terreno|valor_catastral_mejoras|valor_catastral_total|valor_para_impuestos|
+-------+------------------+------------------+-----------------------+-----------------------+---------------------+--------------------+
|count  |1468837           |1468837           |1468837                |1468837                |1468837              |1468837             |
|min    |1                 |0                 |0                      |0                      |1                    |1                   |
|max    |7549858           |418524            |2270351187             |2305649843             |2880092039           |2880092039          |
|mean   |3359.2960941207225|117.88195763042461|309624.3503894578      |576933.3531651231      |1573257.3518184796   |1570875.464580481   |
|stddev |25863.443761428978

In [15]:
df_padrones_rurales_eda = convertir_columnas_numericas(
    dfs["padrones_rurales"],
    columnas_long=[
        "area_predio",
        "valor_catastral_total",
        "valor_para_impuestos"
    ],
    columnas_int=[
        "nro_padron",
        "seccion_catastral"
    ]
)

df_padrones_rurales_eda.select(
    "area_predio",
    "valor_catastral_total",
    "valor_para_impuestos"
).summary("count", "min", "max", "mean", "stddev").show(truncate=False)

+-------+-----------------+---------------------+--------------------+
|summary|area_predio      |valor_catastral_total|valor_para_impuestos|
+-------+-----------------+---------------------+--------------------+
|count  |248022           |248022               |248022              |
|min    |8                |4                    |4                   |
|max    |374918847        |946707581            |946707581           |
|mean   |689947.500770093 |1993472.5505116482   |1993314.1659933394  |
|stddev |2129648.673035917|5833814.69480265     |5833741.592370993   |
+-------+-----------------+---------------------+--------------------+



In [16]:
df_lineas_eda = convertir_columnas_numericas(
    dfs["lineas_de_construccion"],
    columnas_int=[
        "area_construida",
        "anio_construccion",
        "anio_remanente"
    ],
    columnas_double=[
        "nivel",
        "codigo_categoria",
        "codigo_estado"
    ]
)

df_lineas_eda.select(
    "area_construida",
    "anio_construccion",
    "anio_remanente",
    "nivel",
    "codigo_categoria",
    "codigo_estado"
).summary("count", "min", "max", "mean", "stddev").show(truncate=False)

[Stage 393:============================>                            (1 + 1) / 2]

+-------+------------------+------------------+------------------+------------------+------------------+------------------+
|summary|area_construida   |anio_construccion |anio_remanente    |nivel             |codigo_categoria  |codigo_estado     |
+-------+------------------+------------------+------------------+------------------+------------------+------------------+
|count  |4478065           |4478065           |4478065           |4478065           |4478065           |4478065           |
|min    |0                 |0                 |0                 |-101.0            |0.0               |0.0               |
|max    |143982            |3013              |2025              |101.0             |9.0               |9.0               |
|mean   |38.47806809414334 |1945.4464363514153|100.5801333835038 |1.3260131329044476|3.277369712141293 |2.5262310841847984|
|stddev |206.77567593042224|270.9715514260873 |431.65344845465773|3.0208839267562446|0.7401041086896493|1.0755392193965512|
+-------

### Observaciones sobre estadísticas numéricas

En `padrones_urbanos` se observa que el área edificada puede ser cero, lo cual puede corresponder a predios sin construcción registrada. Los valores catastrales presentan mínimos mayores a cero para el valor total y el valor para impuestos, lo que indica que estas medidas están informadas para todos los registros urbanos.

En `padrones_rurales` las áreas promedio y máximas son considerablemente mayores que en padrones urbanos, lo cual es esperable por la naturaleza del dominio rural.

En `lineas_de_construccion` se detectan algunos valores que requieren atención para el refinamiento posterior:

- `anio_construccion = 0`, que se interpretará como año no informado.
- `anio_construccion > 2026`, que se considerará inconsistente para el análisis.
- `area_construida = 0`, que puede representar información no declarada o líneas sin área computable.
- Valores extremos en `nivel`, como -101 y 101, que deberán interpretarse con cautela.
- Códigos de categoría o estado iguales a 0, que pueden representar valores no clasificados o no informados.

Estas observaciones serán consideradas al definir las reglas de limpieza y modelado.

## 10. Detección de valores atípicos o inconsistentes

A partir de las estadísticas básicas se identifican posibles inconsistencias en algunas columnas, especialmente en la tabla de líneas de construcción.

Se analizan años de construcción no informados, años futuros, áreas construidas iguales a cero y niveles extremos.

In [17]:
from pyspark.sql.functions import current_date, year

anio_actual = 2026

df_calidad_lineas = df_lineas_eda.select(
    count("*").alias("total_registros"),
    count(when(col("anio_construccion") == 0, True)).alias("anio_construccion_cero"),
    count(when(col("anio_construccion") > anio_actual, True)).alias("anio_construccion_futuro"),
    count(when(col("area_construida") == 0, True)).alias("area_construida_cero"),
    count(when(col("nivel") < -10, True)).alias("nivel_menor_a_menos_10"),
    count(when(col("nivel") > 50, True)).alias("nivel_mayor_a_50"),
    count(when(col("codigo_categoria") == 0, True)).alias("categoria_cero"),
    count(when(col("codigo_estado") == 0, True)).alias("estado_cero")
)

df_calidad_lineas.show(truncate=False)

[Stage 396:============================>                            (1 + 1) / 2]

+---------------+----------------------+------------------------+--------------------+----------------------+----------------+--------------+-----------+
|total_registros|anio_construccion_cero|anio_construccion_futuro|area_construida_cero|nivel_menor_a_menos_10|nivel_mayor_a_50|categoria_cero|estado_cero|
+---------------+----------------------+------------------------+--------------------+----------------------+----------------+--------------+-----------+
|4478065        |84285                 |8                       |672857              |4                     |102             |10923         |92613      |
+---------------+----------------------+------------------------+--------------------+----------------------+----------------+--------------+-----------+



In [18]:
total_lineas = df_lineas_eda.count()

df_calidad_lineas_porcentaje = df_calidad_lineas.select(
    (col("anio_construccion_cero") / total_lineas * 100).alias("pct_anio_construccion_cero"),
    (col("anio_construccion_futuro") / total_lineas * 100).alias("pct_anio_construccion_futuro"),
    (col("area_construida_cero") / total_lineas * 100).alias("pct_area_construida_cero"),
    (col("nivel_menor_a_menos_10") / total_lineas * 100).alias("pct_nivel_menor_a_menos_10"),
    (col("nivel_mayor_a_50") / total_lineas * 100).alias("pct_nivel_mayor_a_50"),
    (col("categoria_cero") / total_lineas * 100).alias("pct_categoria_cero"),
    (col("estado_cero") / total_lineas * 100).alias("pct_estado_cero")
)

df_calidad_lineas_porcentaje.show(truncate=False)

[Stage 402:============================>                            (1 + 1) / 2]

+--------------------------+----------------------------+------------------------+--------------------------+---------------------+-------------------+------------------+
|pct_anio_construccion_cero|pct_anio_construccion_futuro|pct_area_construida_cero|pct_nivel_menor_a_menos_10|pct_nivel_mayor_a_50 |pct_categoria_cero |pct_estado_cero   |
+--------------------------+----------------------------+------------------------+--------------------------+---------------------+-------------------+------------------+
|1.882174555304579         |1.78648590406794E-4         |15.025619324418024      |8.9324295203397E-5        |0.0022777695276866234|0.24392231912667636|2.0681477379180517|
+--------------------------+----------------------------+------------------------+--------------------------+---------------------+-------------------+------------------+



### Interpretación de valores atípicos o inconsistentes

El análisis de calidad sobre `lineas_de_construccion` muestra que la mayor inconsistencia relevante se encuentra en `area_construida = 0`, con aproximadamente 15% de los registros. Este valor puede indicar construcciones sin área computable, registros incompletos o líneas informativas.

También se detectan años de construcción iguales a cero, que representan aproximadamente 1,88% de los registros. Para el análisis posterior estos valores serán tratados como años no informados.

Los años de construcción mayores al año actual son muy pocos, apenas 8 registros, por lo que se consideran inconsistencias puntuales.

Los niveles extremos menores a -10 o mayores a 50 son muy poco frecuentes. No se eliminarán en esta etapa, pero se tendrán en cuenta para evitar interpretaciones incorrectas.

Los códigos `codigo_categoria = 0` y `codigo_estado = 0` se interpretarán como valores no clasificados o no informados, ya que aparecen en una proporción baja del total.

## 11. Validación de integridad referencial

En esta sección se valida si los códigos utilizados en las tablas principales tienen correspondencia con las tablas auxiliares.

Se revisan relaciones como:

- Padrones urbanos y rurales contra departamentos.
- Padrones urbanos contra localidades.
- Líneas de construcción contra destinos, categorías, estados, cubiertas, cielorrasos y tipos de obra.

Esta validación permite identificar códigos huérfanos o valores que no tienen descripción en las tablas maestras.

In [19]:
def validar_referencia(df_origen, df_dim, columnas_origen, columnas_dim, nombre_validacion):
    """
    Valida si los valores de columnas_origen en df_origen tienen correspondencia
    en columnas_dim de df_dim.
    """
    total_origen = df_origen.count()
    
    df_no_match = (
        df_origen
        .join(
            df_dim,
            on=[df_origen[o] == df_dim[d] for o, d in zip(columnas_origen, columnas_dim)],
            how="left_anti"
        )
    )
    
    cantidad_no_match = df_no_match.count()
    
    return Row(
        validacion=nombre_validacion,
        total_registros_origen=total_origen,
        registros_sin_match=cantidad_no_match,
        porcentaje_sin_match=round((cantidad_no_match / total_origen) * 100, 4) if total_origen > 0 else 0
    )

In [20]:
validaciones_ref = []

# Departamentos
validaciones_ref.append(
    validar_referencia(
        dfs["padrones_urbanos"],
        dfs["departamentos"],
        ["codigo_departamento"],
        ["codigo_departamento"],
        "padrones_urbanos -> departamentos"
    )
)

validaciones_ref.append(
    validar_referencia(
        dfs["padrones_rurales"],
        dfs["departamentos"],
        ["codigo_departamento"],
        ["codigo_departamento"],
        "padrones_rurales -> departamentos"
    )
)

validaciones_ref.append(
    validar_referencia(
        dfs["lineas_de_construccion"],
        dfs["departamentos"],
        ["codigo_departamento"],
        ["codigo_departamento"],
        "lineas_de_construccion -> departamentos"
    )
)

# Localidades
validaciones_ref.append(
    validar_referencia(
        dfs["padrones_urbanos"],
        dfs["localidades"],
        ["codigo_departamento", "codigo_localidad"],
        ["codigo_departamento", "codigo_localidad"],
        "padrones_urbanos -> localidades"
    )
)

validaciones_ref.append(
    validar_referencia(
        dfs["lineas_de_construccion"],
        dfs["localidades"],
        ["codigo_departamento", "codigo_localidad"],
        ["codigo_departamento", "codigo_localidad"],
        "lineas_de_construccion -> localidades"
    )
)

# Tablas auxiliares de líneas de construcción
validaciones_ref.append(
    validar_referencia(
        dfs["lineas_de_construccion"],
        dfs["destinos"],
        ["codigo_destino"],
        ["codigo_destino"],
        "lineas_de_construccion -> destinos"
    )
)

validaciones_ref.append(
    validar_referencia(
        dfs["lineas_de_construccion"],
        dfs["categorias_de_construccion"],
        ["codigo_categoria"],
        ["codigo_categoria"],
        "lineas_de_construccion -> categorias_de_construccion"
    )
)

validaciones_ref.append(
    validar_referencia(
        dfs["lineas_de_construccion"],
        dfs["estados_de_conservacion"],
        ["codigo_estado"],
        ["codigo_estado"],
        "lineas_de_construccion -> estados_de_conservacion"
    )
)

validaciones_ref.append(
    validar_referencia(
        dfs["lineas_de_construccion"],
        dfs["cubiertas"],
        ["codigo_cubierta"],
        ["codigo_cubierta"],
        "lineas_de_construccion -> cubiertas"
    )
)

validaciones_ref.append(
    validar_referencia(
        dfs["lineas_de_construccion"],
        dfs["cielorrasos"],
        ["codigo_cielorraso"],
        ["codigo_cielorraso"],
        "lineas_de_construccion -> cielorrasos"
    )
)

validaciones_ref.append(
    validar_referencia(
        dfs["lineas_de_construccion"],
        dfs["tipos_de_obra"],
        ["codigo_tipo_obra"],
        ["codigo_tipo_obra"],
        "lineas_de_construccion -> tipos_de_obra"
    )
)

df_validaciones_ref = spark.createDataFrame(validaciones_ref)
df_validaciones_ref.orderBy(desc("registros_sin_match")).show(truncate=False)

+----------------------------------------------------+----------------------+-------------------+--------------------+
|validacion                                          |total_registros_origen|registros_sin_match|porcentaje_sin_match|
+----------------------------------------------------+----------------------+-------------------+--------------------+
|lineas_de_construccion -> cielorrasos               |4478065               |4326645            |96.6186             |
|lineas_de_construccion -> estados_de_conservacion   |4478065               |111074             |2.4804              |
|lineas_de_construccion -> cubiertas                 |4478065               |73955              |1.6515              |
|lineas_de_construccion -> categorias_de_construccion|4478065               |10957              |0.2447              |
|lineas_de_construccion -> destinos                  |4478065               |2002               |0.0447              |
|padrones_urbanos -> localidades                

### Interpretación de integridad referencial

La validación de integridad referencial muestra que las relaciones con `departamentos` son completas para padrones urbanos, padrones rurales y líneas de construcción.

También se observa un porcentaje muy bajo de registros sin correspondencia en localidades, destinos, categorías, cubiertas, estados de conservación y tipos de obra. Estos casos se consideran excepciones o valores no clasificados que deberán contemplarse en el modelado.

El caso más relevante es la relación entre `lineas_de_construccion` y `cielorrasos`, donde más del 96% de los registros no tienen correspondencia con la tabla auxiliar. Esto indica que la tabla `cielorrasos` no cubre todos los códigos presentes en las líneas de construcción. En particular, es probable que el código `0` represente ausencia de cielorraso o valor no informado.

Para evitar pérdida de información en el modelo analítico, se deberá crear una dimensión de cielorraso que contemple explícitamente los códigos presentes en la tabla de líneas de construcción, incluyendo valores no informados o no clasificados.

### Revisión de códigos de cielorraso

Dado el alto porcentaje de registros sin correspondencia con la tabla auxiliar de cielorrasos, se revisan los valores distintos presentes en `lineas_de_construccion` y en la tabla `cielorrasos`.

In [21]:
print("Valores en tabla cielorrasos:")
dfs["cielorrasos"].show(truncate=False)

print("Distribución de codigo_cielorraso en lineas_de_construccion:")
(
    dfs["lineas_de_construccion"]
    .groupBy("codigo_cielorraso")
    .count()
    .orderBy(desc("count"))
    .show(20, truncate=False)
)

Valores en tabla cielorrasos:
+-----------------+--------------+
|codigo_cielorraso|cielorraso    |
+-----------------+--------------+
|1                |Con cielorraso|
+-----------------+--------------+

Distribución de codigo_cielorraso en lineas_de_construccion:
+-----------------+-------+
|codigo_cielorraso|count  |
+-----------------+-------+
|0                |4326645|
|1                |151420 |
+-----------------+-------+



### Resultado de la revisión de cielorrasos

La tabla auxiliar `cielorrasos` contiene únicamente el código `1`, asociado a la descripción "Con cielorraso".

Sin embargo, en la tabla `lineas_de_construccion` existen dos valores distintos para `codigo_cielorraso`:

- `0`: 4.326.645 registros.
- `1`: 151.420 registros.

Esto explica el alto porcentaje de registros sin correspondencia en la validación referencial. El código `0` no está presente en la tabla auxiliar, pero aparece de forma masiva en la tabla principal.

Para el modelo analítico se interpretará el código `0` como "Sin cielorraso / No informado" y se construirá una dimensión enriquecida que incluya ambos códigos.

### Decisión para el modelado

La tabla `cielorrasos` se conservará como tabla auxiliar original en la zona refinada.  
Sin embargo, para el modelo analítico se deberá construir una dimensión corregida o enriquecida, agregando una descripción para los códigos presentes en las líneas de construcción que no figuran en la tabla auxiliar.

Esta decisión evita descartar registros de construcción y permite mantener la trazabilidad entre el dato original y el modelo analítico.

## 12. Distribución de valores en tablas auxiliares

En esta sección se revisan los valores disponibles en las tablas auxiliares del dominio catastral.  
Estas tablas permiten interpretar códigos presentes en las tablas principales, especialmente en `lineas_de_construccion`.

Las tablas auxiliares luego serán candidatas a dimensiones en el modelo analítico.

In [22]:
tablas_auxiliares = [
    "departamentos",
    "destinos",
    "categorias_de_construccion",
    "estados_de_conservacion",
    "cubiertas",
    "cielorrasos",
    "tipos_de_obra"
]

for tabla in tablas_auxiliares:
    print("=" * 80)
    print(f"TABLA AUXILIAR: {tabla}")
    print("=" * 80)
    dfs[tabla].show(100, truncate=False)

TABLA AUXILIAR: departamentos
+-------------------+--------------+
|codigo_departamento|departamento  |
+-------------------+--------------+
|A                  |CANELONES     |
|B                  |MALDONADO     |
|C                  |ROCHA         |
|D                  |TREINTA Y TRES|
|E                  |CERRO LARGO   |
|F                  |RIVERA        |
|G                  |ARTIGAS       |
|H                  |SALTO         |
|I                  |PAYSANDU      |
|J                  |RIO NEGRO     |
|K                  |SORIANO       |
|L                  |COLONIA       |
|M                  |SAN JOSE      |
|N                  |FLORES        |
|O                  |FLORIDA       |
|P                  |LAVALLEJA     |
|Q                  |DURAZNO       |
|R                  |TACUAREMBO    |
|V                  |MONTEVIDEO    |
+-------------------+--------------+

TABLA AUXILIAR: destinos
+--------------+--------------------------+
|codigo_destino|destino                   |
+----

### Observación sobre tablas auxiliares

Las tablas auxiliares contienen valores descriptivos que permiten interpretar códigos utilizados en las tablas principales.

Estas tablas serán utilizadas como base para construir dimensiones del modelo analítico. En algunos casos será necesario enriquecerlas, como ocurre con `cielorrasos`, donde el código `0` aparece en `lineas_de_construccion` pero no existe en la tabla auxiliar original.

Mantener las tablas auxiliares originales en la zona refinada permite conservar trazabilidad, mientras que las correcciones o enriquecimientos se realizarán posteriormente en la zona modelada.

### Observaciones sobre tablas auxiliares

Las tablas auxiliares permiten interpretar los códigos utilizados en las tablas principales.

Se observa que:

- `departamentos` contiene los 19 departamentos del país.
- `destinos` contiene 72 códigos de destino de construcción.
- `categorias_de_construccion` contiene categorías principales e intermedias; algunas categorías intermedias no tienen descripción textual.
- `estados_de_conservacion` contiene estados principales e intermedios.
- `cubiertas` contiene los tipos de cubierta.
- `cielorrasos` contiene únicamente el código `1`, por lo que deberá enriquecerse en el modelo analítico incorporando el código `0`.
- `tipos_de_obra` contiene códigos que agrupan obras originales, reformas, obras paralizadas, habilitadas sin terminar, a construir y a demoler.

También se detectan algunos problemas de codificación de caracteres en textos descriptivos, por ejemplo en palabras con ñ o tildes. Estos problemas no afectan las claves ni los cálculos principales, pero se tendrán en cuenta en la presentación del informe.

## 13. Conclusiones del análisis exploratorio

A partir del análisis exploratorio realizado sobre las tablas refinadas de Catastro, se obtienen las siguientes conclusiones:

### Tablas principales

Las tablas principales para el modelo analítico serán:

- `padrones_urbanos`
- `padrones_rurales`
- `lineas_de_construccion`
- `historico_de_valores`

Estas tablas concentran la información de mayor volumen y las principales medidas del dominio, como áreas, valores catastrales, años de construcción y características constructivas.

### Tablas auxiliares

Las tablas auxiliares serán utilizadas como dimensiones descriptivas:

- `departamentos`
- `localidades`
- `destinos`
- `categorias_de_construccion`
- `estados_de_conservacion`
- `cubiertas`
- `cielorrasos`
- `tipos_de_obra`

### Calidad de datos

Se identificaron los siguientes aspectos relevantes:

- Las tablas de departamentos, localidades y catálogos principales presentan claves únicas.
- `padrones_urbanos`, `padrones_rurales` e `historico_de_valores` presentan claves candidatas únicas.
- `lineas_de_construccion` presenta duplicados según la clave candidata propuesta, por lo que se utilizará como tabla de hechos sin asumir una clave primaria estricta.
- `mutaciones_catastrales` presenta duplicados y posibles problemas de interpretación de columnas, por lo que no será utilizada como tabla principal del modelo analítico.
- En `lineas_de_construccion` existen valores de `anio_construccion = 0`, que serán interpretados como no informados.
- Existen algunos años de construcción futuros, que serán tratados como inconsistentes.
- El código `codigo_cielorraso = 0` aparece masivamente en líneas de construcción, pero no existe en la tabla auxiliar `cielorrasos`. Para el modelo se incorporará como "Sin cielorraso / No informado".
- Algunas descripciones presentan problemas de codificación de caracteres, pero las claves se encuentran disponibles para realizar los cruces.

### Decisiones para la siguiente etapa

En la etapa de modelado se deberán aplicar las siguientes decisiones:

1. Convertir columnas numéricas desde `string` a tipos numéricos adecuados.
2. Construir dimensiones limpias y enriquecidas.
3. Enriquecer la dimensión `cielorrasos` incorporando el código `0`.
4. Tratar `anio_construccion = 0` como valor no informado.
5. Evitar utilizar años futuros para análisis de antigüedad.
6. Usar `lineas_de_construccion` como tabla de hechos de construcciones.
7. Utilizar `padrones_urbanos`, `padrones_rurales` e `historico_de_valores` para análisis de áreas y valores catastrales.